### Gold Incremental 
Incremental gold processing plus latest and timestamped Volume snapshots

### Step 1 - Imports and Setup
This cell imports the required helpers, switches to the right catalog, makes sure the Gold schema exists and creates a 

- a gold_run_id
- a run date string
- a run timestamp string

These are used for tracking and snapshot publishing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
# Set catalog and schema
spark.sql("use catalog novacart_databricks")
spark.sql("create schema if not exists gold_schema")

# Initialize run variables
gold_run_id = str(uuid.uuid4())
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print(f"Current Gold Run ID: {gold_run_id}")
print(f"Run Timestamp: {run_ts_str}")

### Step 2 - Gold Control Table 

This table stores the latest gold processing state

It tells Gold : 
- which silver data was processed last time 
- how many gold rows were merged in the last run


In [0]:
spark.sql("""
create table if not exists novacart_databricks.gold_schema.processing_control (
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
)
using delta
""")

### Step 3 - Helper Functions
This cell defines reusable gold functions : 

- upser_to_gold() merges daat into gold current-state tables
- get_last_processed_silver_ts() reads the gold watermark from the control table
- upsert_gold_control() updates gold control after a successful run

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    """Merges source data into the Gold target table."""
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
def get_last_processed_silver_ts(entity_name: str):
    """Reads the Gold watermark from the control table."""
    ctrl = (spark.table("novacart_databricks.gold_schema.processing_control")
            .filter((F.col("layer") == "gold") & 
                    (F.col("entity_name") == entity_name) & 
                    (F.col("run_status") == "SUCCESS"))
            .orderBy(F.col("updated_at").desc())
            .limit(1))
    
    rows = ctrl.collect()
    if not rows:
        return
    return rows[0]["last_processed_silver_run_ts"] 


In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name, last_processed_silver_run_id, last_processed_run_ts, 
            int(rows_merged), 
            "SUCCESS", 
            gold_run_id, 
            datetime.utcnow()
            )],
        schema = """
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "novacart_databricks.gold_schema.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
        .whenMatchedUpdate(set= {
            "last_processed_silver_run_id": "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


### Step 4 - Read changed silver rows only 

This cell reads the full silver current-state tables but filters out only the rows that changed since the last gold run. 

This is the starting point for Gold incremental processing.

In [0]:
# Example for 'orders_information'
last_gold_ts = get_last_processed_silver_ts("orders_information")
print(f"Last Processed Silver Timestamp for Gold =", last_gold_ts)

# Load current Silver tables
silver_orders_current = spark.read.table("novacart_databricks.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_databricks.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_databricks.silver_schema.payments_transformed")

if last_gold_ts is None:
    # Full load if no watermark exists
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    # Incremental filter
    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("processed_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()


# Summary counts
print("Number of changed orders:", changed_orders_count)
print("Number of changed products:", changed_products_count)
print("Number of changed payments:", changed_payments_count)

### Step 5 - Find impacted order IDs
Gold is built at order grain so if anything changes in orders, products, or payments we identify which order_id values are impacted.

Only those order IDs are rebuilt in Gold

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()

impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"), 
          F.col("p.product_id") == F.col("o.product_id"), 
          "inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print(f"Impacted_order_ids = {impacted_order_ids.count()}")
display(impacted_order_ids.orderBy("order_id"))


### Step 6 - Build Gold delta for impacted orders

This cell joins the impacted orders with the current silver products and payments tables, derives business columns and builds the gold delta that will be merged into the Gold current-state table

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(silver_products_current.alias("p"), 
          F.col("o.product_id") == F.col("p.product_id"), 
          "inner")
    .join(silver_payments_current.alias("py"), 
          F.col("o.order_id") == F.col("py.order_id"), 
          "inner")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("products_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn("payment_completion_ratio", 
                F.when(F.col("order_amount") > 0, 
                       F.col("paid_amount") / F.col("order_amount"))
                .otherwise(F.lit(0.0)))
    .withColumn("payment_state",
                F.when(F.col("order_amount") == 0, "Invalid_order_amount")
                .when(F.col("payment_completion_ratio") == 0, "Unpaid")
                .when(F.col("payment_completion_ratio") == 1, "Paid")
                .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
                .when(F.col("payment_completion_ratio") > 1, "Overpaid"))
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

print(f"gold_delta = {gold_delta.count()}")
display(gold_delta)

### Step 7 - Merge Gold current-state table
If Gold delta contains rows, this cell merges them into gold_schema.orders_information.

If there are no impacted rows, nothing is merged

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novacart_databricks.gold_schema.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")

In [0]:
%sql
select * from novacart_databricks.gold_schema.orders_information;

### Step 8 - Maintain Gold SCD Type 2 history

This cell updates the SCD type 2 history table

If a current Gold row changes, the old version is closed(is_current = false) and a new current version is inserted.

In [0]:
# Initialize SCD2 table if it doesn't exist
if not spark.catalog.tableExists("novacart_databricks.gold_schema.orders_information_scd2"):
    spark.sql("""
        create table novacart_databricks.gold_schema.orders_information_scd2
        using delta
        as select *, 
           cast(null as timestamp) as valid_from_ts,
           cast(null as timestamp) as valid_to_ts,
           true as is_current
        from gold_schema.orders_information where 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    
    # Close old versions
    spark.sql("""
        merge into novacart_databricks.gold_schema.orders_information_scd2 t
        using gold_delta_view s
        on t.order_id = s.order_id and t.is_current = true
        when matched and (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.products_price <=> s.products_price)
        )
        then update set 
            t.valid_to_ts = s.gold_update_ts, 
            t.is_current = false
    """)
    
    # Insert new versions
    spark.sql("""
        insert into novacart_databricks.gold_schema.orders_information_scd2
        select s.*, 
            s.gold_update_ts as valid_from_ts, 
            cast(null as timestamp) as valid_to_ts, true as is_current
        from gold_delta_view s
        left join novacart_databricks.gold_schema.orders_information_scd2 t
        on s.order_id = t.order_id
        and t.is_current = true
        where t.order_id is null or (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or 
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.products_price <=> s.products_price)
        )
    """)

### Step 9 - Update category-level gold aggregation 

This cell recalculates category-level business metrics only for categories impacted in the current run, then merges them into the category performance gold table

In [0]:
if gold_delta.count() > 0:
    # Identify which categories have new/changed data
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    # Recalculate metrics for those specific categories
    category_perf_delta = (
        spark.read.table("novacart_databricks.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg(F.col("payment_completion_ratio")).alias("Average_Payment_Completion_Ratio"),
            (F.sum(F.when(F.col("payment_status") == "FAILED", 1).otherwise(0)) / F.count("*")).alias("Payment_Failure_Rate")
        )
    )

    # Save to the performance table
    upsert_to_gold(category_perf_delta, "gold_schema.category_performance", "category")

# Verification SQL
# %sql
# select * from gold_schema.category_performance;

In [0]:
%sql 
select * from novacart_databricks.gold_schema.category_performance;

### Step 10 - Publish gold snapshots to Volume

This cell writes two kinds of gold outputs to a Databricks Volume:
  - latest snapshot - overwritten every successful run
  - timestamped historcial snapshot - a new folder for each successful run

This is useful for audit, rollback, and teaching demos

In [0]:
# Create the volume if it doesn't exist
spark.sql("create volume if not exists novacart_databricks.gold_schema.gold_snapshots_vol")


In [0]:
# Define storage paths
volume_path = "/Volumes/novacart_databricks/gold_schema/gold_snapshots_vol"
latest_orders_path = f"{volume_path}/gold_latest/orders_information"
latest_category_path = f"{volume_path}/gold_latest/category_performance"
historical_orders_path = f"{volume_path}/gold_historical/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path = f"{volume_path}/gold_historical/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

# Overwrite 'Latest' snapshot
(spark.read.table("novacart_databricks.gold_schema.orders_information")
 .write.format("parquet").mode("overwrite").save(latest_orders_path))

(spark.read.table("novacart_databricks.gold_schema.category_performance")
 .write.format("parquet").mode("overwrite").save(latest_category_path))

# Save 'Historical' timestamped snapshot
(spark.read.table("novacart_databricks.gold_schema.orders_information")
 .write.format("parquet").mode("overwrite").save(historical_orders_path))

(spark.read.table("novacart_databricks.gold_schema.category_performance")
 .write.format("parquet").mode("overwrite").save(historical_category_path))

print("latest orders path :", latest_orders_path)
print("latest category path :", latest_category_path)
print("historical orders path :", historical_orders_path)
print("historical category path :", historical_category_path)

### Step 11 - Update gold control table

This final cell updates the gold control table using the latest silver processing metadata and displays the control table for validation

In [0]:
# Get the latest silver timestamp that was processed
latest_silver_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

# Get the corresponding silver run ID
latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

# Update the control table for the next run
upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

# Display control table for validation
display(spark.table("novacart_databricks.gold_schema.processing_control"))

In [0]:
/Volumes/novacart_databricks/gold_schema/gold_snapshots_vol/gold_historical//Volumes/novacart_databricks/gold_schema/gold_snapshots_vol/gold_latest//Volumes/novacart_databricks/gold_schema/gold_snapshots_vol/gold_latest/